# Notebook 02 — Four-File Integration & Temporal Recommendation Task

**MVP goal:** validate both property parts and the category tree, create point-in-time item features, and export leakage-aware train/validation/test user–item examples. No model training yet.

**Important:** RetailRocket logs *observed interactions*, not ad impressions. Sampled unobserved pairs are **weak negatives**, not confirmed dislikes. This is an offline recommendation proxy, not true CTR prediction.

Run top to bottom from the repository root or `notebooks/` in Windows VS Code/Jupyter. Expected raw files: `events.csv`, `item_properties_part1.csv`, `item_properties_part2.csv`, `category_tree.csv` in `data/raw/`.

Install once: `python -m pip install duckdb pandas numpy pyarrow ipykernel`

In [2]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import duckdb

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'data' / 'raw' / 'events.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Cannot locate data/raw/events.csv. Open VS Code at the project root or notebooks folder.')
RAW, OUT, REPORT = ROOT/'data'/'raw', ROOT/'data'/'processed', ROOT/'data'/'reports'
OUT.mkdir(parents=True, exist_ok=True); REPORT.mkdir(parents=True, exist_ok=True)
paths = {k: RAW/f for k,f in {'events':'events.csv','p1':'item_properties_part1.csv','p2':'item_properties_part2.csv','tree':'category_tree.csv'}.items()}
for k,p in paths.items():
    assert p.is_file(), f'Missing {p}'
con = duckdb.connect(str(OUT/'notebook02.duckdb'))
con.execute("SET memory_limit='4GB'") # Adjust downward for low-RAM PCs
con.execute("SET threads=4")
con.execute(f"SET temp_directory='{(OUT/'duckdb_tmp').as_posix()}'")
q = lambda p: "'" + p.as_posix().replace("'", "''") + "'"
print('Project:', ROOT)
print('DuckDB:', duckdb.__version__)

Project: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks
DuckDB: 1.5.5


## A1. Read and validate four schemas
Use explicit types and `read_csv(..., header=true)` to avoid costly type inference. `transactionid` is nullable. Timestamps are Unix milliseconds UTC.

In [3]:
con.execute(f"""CREATE OR REPLACE VIEW events_raw AS SELECT timestamp::BIGINT AS ts, visitorid::BIGINT AS user_id, event::VARCHAR AS event, itemid::BIGINT AS item_id, TRY_CAST(transactionid AS BIGINT) AS transaction_id FROM read_csv({q(paths['events'])}, header=true, columns={{'timestamp':'BIGINT','visitorid':'BIGINT','event':'VARCHAR','itemid':'BIGINT','transactionid':'VARCHAR'}})""")
for key in ('p1','p2'):
    con.execute(f"""CREATE OR REPLACE VIEW {key}_raw AS SELECT timestamp::BIGINT AS ts, itemid::BIGINT AS item_id, property::VARCHAR AS property, value::VARCHAR AS value FROM read_csv({q(paths[key])}, header=true, columns={{'timestamp':'BIGINT','itemid':'BIGINT','property':'VARCHAR','value':'VARCHAR'}})""")
con.execute(f"""CREATE OR REPLACE VIEW category_tree AS SELECT categoryid::BIGINT AS category_id, TRY_CAST(parentid AS BIGINT) AS parent_id FROM read_csv({q(paths['tree'])}, header=true, columns={{'categoryid':'BIGINT','parentid':'VARCHAR'}})""")
con.execute('CREATE OR REPLACE VIEW props_all AS SELECT * FROM p1_raw UNION ALL SELECT * FROM p2_raw')
for view in ['events_raw','p1_raw','p2_raw','category_tree']:
    print(view, con.execute(f'SELECT COUNT(*) FROM {view}').fetchone()[0], con.execute(f'DESCRIBE {view}').df()[['column_name','column_type']].to_dict('records'))

events_raw 2756101 [{'column_name': 'ts', 'column_type': 'BIGINT'}, {'column_name': 'user_id', 'column_type': 'BIGINT'}, {'column_name': 'event', 'column_type': 'VARCHAR'}, {'column_name': 'item_id', 'column_type': 'BIGINT'}, {'column_name': 'transaction_id', 'column_type': 'BIGINT'}]
p1_raw 10999999 [{'column_name': 'ts', 'column_type': 'BIGINT'}, {'column_name': 'item_id', 'column_type': 'BIGINT'}, {'column_name': 'property', 'column_type': 'VARCHAR'}, {'column_name': 'value', 'column_type': 'VARCHAR'}]
p2_raw 9275903 [{'column_name': 'ts', 'column_type': 'BIGINT'}, {'column_name': 'item_id', 'column_type': 'BIGINT'}, {'column_name': 'property', 'column_type': 'VARCHAR'}, {'column_name': 'value', 'column_type': 'VARCHAR'}]
category_tree 1669 [{'column_name': 'category_id', 'column_type': 'BIGINT'}, {'column_name': 'parent_id', 'column_type': 'BIGINT'}]


## A2. Cross-part exact duplicates and logical property changes
Identical `(ts,item_id,property,value)` rows are exact duplicates. Repeated `(item_id,property)` at different timestamps are often legitimate updates; do **not** drop them. Same `(ts,item_id,property)` with different values is a conflict requiring deterministic handling.

In [4]:
t0=time.perf_counter()
prop_stats=con.execute("""
WITH grouped AS (
 SELECT ts,item_id,property, COUNT(*) AS n, COUNT(DISTINCT value) AS distinct_values
 FROM props_all GROUP BY 1,2,3
), exacts AS (
 SELECT ts,item_id,property,value,COUNT(*) n FROM props_all GROUP BY 1,2,3,4
)
SELECT
 (SELECT SUM(n-1) FROM exacts WHERE n>1) AS exact_duplicate_excess_rows,
 (SELECT COUNT(*) FROM grouped WHERE distinct_values>1) AS conflicting_timestamp_keys,
 (SELECT COUNT(*) FROM grouped WHERE n>1) AS repeated_timestamp_keys
""").df().iloc[0].to_dict()
updates=con.execute("""SELECT COUNT(*) AS item_property_pairs_with_multiple_timestamps FROM (SELECT item_id,property FROM props_all GROUP BY 1,2 HAVING COUNT(DISTINCT ts)>1)""").fetchone()[0]
prop_stats['item_property_pairs_with_multiple_timestamps']=int(updates)
print(prop_stats, 'elapsed seconds',round(time.perf_counter()-t0,1))

{'exact_duplicate_excess_rows': nan, 'conflicting_timestamp_keys': 0.0, 'repeated_timestamp_keys': 0.0, 'item_property_pairs_with_multiple_timestamps': 514679} elapsed seconds 16.6


## A3. Category relationships and coverage
Only a property literally named `categoryid` is considered an item-to-category mapping. Other properties are not guessed to be categories. Validate category parent references; root nodes may have null parents.

In [5]:
category_property_names=con.execute("SELECT property,COUNT(*) AS n FROM props_all WHERE lower(property) LIKE '%categor%' GROUP BY 1 ORDER BY n DESC").df()
print('Category-like property names:'); display(category_property_names.head(20))
tree_checks = con.execute("""
    SELECT
        COUNT(*) AS nodes,

        COUNT(*) FILTER (
            WHERE t.parent_id IS NULL
        ) AS roots,

        COUNT(*) FILTER (
            WHERE t.parent_id IS NOT NULL
              AND p.category_id IS NULL
        ) AS orphan_parent_links,

        COUNT(*) FILTER (
            WHERE t.category_id = t.parent_id
        ) AS self_parent_links

    FROM category_tree t

    LEFT JOIN category_tree p
        ON t.parent_id = p.category_id
""").df().iloc[0].to_dict()
print('Tree checks:',tree_checks)
cat_stats=con.execute("""WITH mapping AS (
 SELECT item_id, TRY_CAST(value AS BIGINT) AS category_id
 FROM props_all WHERE property='categoryid'
), e AS (SELECT DISTINCT item_id FROM events_raw),
 valid_items AS (SELECT DISTINCT m.item_id FROM mapping m JOIN category_tree t USING(category_id)),
 prop_items AS (SELECT DISTINCT item_id FROM props_all)
SELECT (SELECT COUNT(*) FROM e) AS event_items,
 (SELECT COUNT(*) FROM e JOIN prop_items USING(item_id)) AS event_items_with_properties,
 (SELECT COUNT(*) FROM e JOIN valid_items USING(item_id)) AS event_items_with_valid_category,
 (SELECT COUNT(*) FROM mapping WHERE category_id IS NULL) AS unparseable_category_rows,
 (SELECT COUNT(*) FROM mapping m LEFT JOIN category_tree t USING(category_id) WHERE m.category_id IS NOT NULL AND t.category_id IS NULL) AS unknown_category_rows
""").df().iloc[0].to_dict()
cat_stats={k:int(v) for k,v in cat_stats.items()}
cat_stats['property_coverage_fraction']=cat_stats['event_items_with_properties']/max(1,cat_stats['event_items'])
cat_stats['category_coverage_fraction']=cat_stats['event_items_with_valid_category']/max(1,cat_stats['event_items'])
print(cat_stats)

Category-like property names:


,property,n
0,categoryid,788214


Tree checks: {'nodes': 1669, 'roots': 25, 'orphan_parent_links': 0, 'self_parent_links': 0}
{'event_items': 235061, 'event_items_with_properties': 185246, 'event_items_with_valid_category': 185246, 'unparseable_category_rows': 0, 'unknown_category_rows': 167, 'property_coverage_fraction': 0.7880762865809301, 'category_coverage_fraction': 0.7880762865809301}


## B1. Task and chronological cutoffs
Each snapshot at time **t** uses 30 days of historical events and predicts any **view/addtocart/transaction** interaction in `(t,t+7 days]`. Cutoffs are at 65%, 78%, and 90% of the full event time span, leaving nonoverlapping 7-day label windows (asserted). Each split contains **one snapshot** to keep the MVP manageable. This is a chronological evaluation design, not an unbiased online experiment.

**Item universe:** items observed before t. **Candidates:** all previously seen items except the user's history items (for the MVP). The same user may occur across snapshots; this is intentional temporal evaluation, not random row splitting.

In [6]:
bounds=con.execute('SELECT MIN(ts),MAX(ts) FROM events_raw').fetchone()
start,end=map(int,bounds)
DAY=86_400_000
cutoffs={name:int(start+(end-start)*fraction) for name,fraction in [('train',.65),('validation',.78),('test',.90)]}
assert cutoffs['train']+7*DAY<cutoffs['validation'] and cutoffs['validation']+7*DAY<cutoffs['test']
assert cutoffs['test']+7*DAY<=end
for name,t in cutoffs.items(): print(name,pd.to_datetime(t,unit='ms',utc=True),'->',pd.to_datetime(t+7*DAY,unit='ms',utc=True))
print('Event counts:',con.execute('SELECT event,COUNT(*) AS n FROM events_raw GROUP BY 1 ORDER BY n DESC').df())

train 2015-07-31 19:47:53.596000+00:00 -> 2015-08-07 19:47:53.596000+00:00
validation 2015-08-18 18:21:27.439000+00:00 -> 2015-08-25 18:21:27.439000+00:00
test 2015-09-04 07:47:49.447000+00:00 -> 2015-09-11 07:47:49.447000+00:00
Event counts:          event        n
0         view  2664312
1    addtocart    69332
2  transaction    22457


## B2. Point-in-time item properties
For each cutoff, take the latest known value per `(item_id,property)` at or before t. When there are conflicting values at the same timestamp, resolve with lexicographically greatest value as a **reproducible fallback**, and report conflicts above; this is not evidence that the chosen value is correct. The exported MVP feature is the category plus property count, not the entire wide property matrix.

In [7]:
def build_snapshot(name,t):
    tag=name.replace('-','_')
    con.execute(f"""CREATE OR REPLACE TABLE {tag}_item_props AS
    WITH latest AS (
      SELECT item_id, property, value, ts,
        ROW_NUMBER() OVER(PARTITION BY item_id,property ORDER BY ts DESC,value DESC) AS rn
      FROM props_all WHERE ts <= {t}
    )
    SELECT item_id,
       COUNT(*) AS known_property_count,
       MAX(CASE WHEN property='categoryid' THEN TRY_CAST(value AS BIGINT) END) AS category_id
    FROM latest WHERE rn=1 GROUP BY item_id""")
    # A category is only considered valid if present in the tree.
    con.execute(f"""CREATE OR REPLACE TABLE {tag}_item_features AS
       SELECT p.item_id,p.known_property_count,
              CASE WHEN tr.category_id IS NOT NULL THEN p.category_id ELSE NULL END AS category_id,
              tr.parent_id AS parent_category_id
       FROM {tag}_item_props p LEFT JOIN category_tree tr ON p.category_id=tr.category_id""")
    con.execute(f"COPY {tag}_item_features TO {q(OUT/f'{name}_item_features.parquet')} (FORMAT PARQUET)")
    print(name,'item features',con.execute(f'SELECT COUNT(*) FROM {tag}_item_features').fetchone()[0])
for name,t in cutoffs.items(): build_snapshot(name,t)

train item features 411127
validation item features 413168
test item features 414949


## B3. Build historical user/item aggregates, positives and sampled weak negatives
To keep a Windows laptop responsive, cap training examples to **50,000 positive user-item pairs per split** using a seeded deterministic hash. Sample up to 3 historical-unobserved items per positive. This deliberately produces a **sampled candidate classification dataset**, not an exhaustive ranking benchmark; later retrieval evaluation must use a larger realistic candidate pool.

**No future leakage:** histories and item popularity use events `< t`; positives use `(t,t+7d]`; item features use property timestamps `<=t`.

In [8]:
MAX_POS=50_000
NEG_PER_POS=3
for name,t in cutoffs.items():
    tag=name
    history_start=t-30*DAY
    con.execute(f"""CREATE OR REPLACE TABLE {tag}_history AS
      SELECT user_id,item_id,COUNT(*) AS n_events,MAX(ts) AS last_ts
      FROM events_raw WHERE ts >= {history_start} AND ts < {t}
      GROUP BY 1,2""")
    con.execute(f"""CREATE OR REPLACE TABLE {tag}_user_features AS
      SELECT user_id,SUM(n_events) AS user_30d_events,COUNT(*) AS user_30d_unique_items,
             {t}-MAX(last_ts) AS user_recency_ms
      FROM {tag}_history GROUP BY 1""")
    con.execute(f"""CREATE OR REPLACE TABLE {tag}_item_popularity AS
      SELECT item_id,SUM(n_events) AS item_30d_events
      FROM {tag}_history GROUP BY 1""")
    # Only users with 30d history and previously observed items; exclude items already seen by that user.
    con.execute(f"""CREATE OR REPLACE TABLE {tag}_positives AS
      WITH future_pairs AS (
        SELECT DISTINCT user_id,item_id FROM events_raw
        WHERE ts > {t} AND ts <= {t+7*DAY}
      ), eligible AS (
        SELECT f.user_id,f.item_id FROM future_pairs f
        JOIN {tag}_user_features u USING(user_id)
        JOIN {tag}_item_popularity i USING(item_id)
        LEFT JOIN {tag}_history h ON f.user_id=h.user_id AND f.item_id=h.item_id
        WHERE h.item_id IS NULL
      )
      SELECT user_id,item_id,1 AS label FROM eligible
      ORDER BY hash(user_id,item_id,42) LIMIT {MAX_POS}""")
    # Deterministic hash-based weak negatives from pre-cutoff popular items.
    con.execute(f"""CREATE OR REPLACE TABLE {tag}_negatives AS
      WITH popular AS (SELECT item_id FROM {tag}_item_popularity ORDER BY item_30d_events DESC LIMIT 1000),
      proposed AS (
        SELECT p.user_id, i.item_id, ROW_NUMBER() OVER(PARTITION BY p.user_id,p.item_id ORDER BY hash(p.user_id,p.item_id)) AS dup
        FROM {tag}_positives p CROSS JOIN LATERAL (
          SELECT item_id FROM popular ORDER BY hash(p.user_id,p.item_id,hash(item_id)) LIMIT {NEG_PER_POS*4}
        ) i
      ), eligible AS (
        SELECT DISTINCT x.user_id,x.item_id FROM proposed x
        LEFT JOIN {tag}_history h ON x.user_id=h.user_id AND x.item_id=h.item_id
        LEFT JOIN {tag}_positives pos ON x.user_id=pos.user_id AND x.item_id=pos.item_id
        WHERE h.item_id IS NULL AND pos.item_id IS NULL
      ), ranked AS (
        SELECT *,ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY hash(user_id,item_id,99)) AS r
        FROM eligible
      )
      SELECT user_id,item_id,0 AS label FROM ranked WHERE r <= {NEG_PER_POS*4}""")
    # Cap weak negatives to ~3 per positive globally, deterministic ordering.
    npos=con.execute(f'SELECT COUNT(*) FROM {tag}_positives').fetchone()[0]
    con.execute(f"""CREATE OR REPLACE TABLE {tag}_examples AS
      WITH negatives AS (SELECT * FROM {tag}_negatives ORDER BY hash(user_id,item_id,2026) LIMIT {npos*NEG_PER_POS}),
      pairs AS (SELECT * FROM {tag}_positives UNION ALL SELECT * FROM negatives)
      SELECT p.user_id,p.item_id,p.label,{t}::BIGINT AS cutoff_ts,
        u.user_30d_events,u.user_30d_unique_items,u.user_recency_ms,
        i.item_30d_events,COALESCE(f.known_property_count,0) AS known_property_count,
        f.category_id,f.parent_category_id
      FROM pairs p JOIN {tag}_user_features u USING(user_id)
      JOIN {tag}_item_popularity i USING(item_id)
      LEFT JOIN {tag}_item_features f USING(item_id)""")
    con.execute(f"COPY {tag}_examples TO {q(OUT/f'{name}_examples.parquet')} (FORMAT PARQUET)")
    print(name,con.execute(f'SELECT COUNT(*) AS rows,SUM(label) AS positives,COUNT(DISTINCT user_id) AS users FROM {tag}_examples').df().to_dict('records'))

train [{'rows': 37448, 'positives': 9362.0, 'users': 4110}]
validation [{'rows': 36080, 'positives': 9020.0, 'users': 3657}]
test [{'rows': 32856, 'positives': 8214.0, 'users': 3572}]


## B4. Sanity checks and exports
All examples must be unique user–item pairs within a split. Snapshot features must be drawn from the historical side of the cutoff. The labels of validation/test snapshots are *not* used to construct their features; the history may include earlier interactions, which would be available in real chronological serving.

In [9]:
report={'cutoffs_utc':{k:str(pd.to_datetime(t,unit='ms',utc=True)) for k,t in cutoffs.items()},
        'property_checks': {
            k: None if pd.isna(v) else int(v)
            for k, v in prop_stats.items()
        },
        'tree_checks':{k:int(v) for k,v in tree_checks.items()},
        'coverage_checks':cat_stats,'splits':{}}
for name,t in cutoffs.items():
    count,positive,unique_pairs=con.execute(f"SELECT COUNT(*),SUM(label),COUNT(DISTINCT (user_id,item_id)) FROM {name}_examples").fetchone()
    assert count==unique_pairs, f'Duplicate pairs in {name}'
    assert count>0 and positive>0, f'Empty or no positives in {name}'
    assert con.execute(f'SELECT COUNT(*) FROM {name}_examples WHERE cutoff_ts != {t}').fetchone()[0]==0
    assert con.execute(f'SELECT COUNT(*) FROM {name}_item_props WHERE known_property_count<1').fetchone()[0]==0
    report['splits'][name]={'examples':int(count),'positives':int(positive),'weak_negatives':int(count-positive)}
with open(REPORT/'notebook02_integration_temporal_report.json','w',encoding='utf-8') as f:json.dump(report,f,indent=2)
print(json.dumps(report,indent=2)[:4500])
print('Saved parquet files to:',OUT)
print('Saved report:',REPORT/'notebook02_integration_temporal_report.json')

{
  "cutoffs_utc": {
    "train": "2015-07-31 19:47:53.596000+00:00",
    "validation": "2015-08-18 18:21:27.439000+00:00",
    "test": "2015-09-04 07:47:49.447000+00:00"
  },
  "property_checks": {
    "exact_duplicate_excess_rows": null,
    "conflicting_timestamp_keys": 0,
    "repeated_timestamp_keys": 0,
    "item_property_pairs_with_multiple_timestamps": 514679
  },
  "tree_checks": {
    "nodes": 1669,
    "roots": 25,
    "orphan_parent_links": 0,
    "self_parent_links": 0
  },
  "coverage_checks": {
    "event_items": 235061,
    "event_items_with_properties": 185246,
    "event_items_with_valid_category": 185246,
    "unparseable_category_rows": 0,
    "unknown_category_rows": 167,
    "property_coverage_fraction": 0.7880762865809301,
    "category_coverage_fraction": 0.7880762865809301
  },
  "splits": {
    "train": {
      "examples": 37448,
      "positives": 9362,
      "weak_negatives": 28086
    },
    "validation": {
      "examples": 36080,
      "positives": 9020,


## Limitations and next milestone
- `categoryid` coverage is **measured**, never assumed. If absent, category features remain null.
- Property timestamps start later than some event timestamps; no backward filling from the future.
- The chosen same-timestamp conflict rule is deterministic but not semantically authoritative.
- The sample-based negatives are unobserved, not proven negatives. The sample label balance cannot be interpreted as real-world CTR or CVR.
- This MVP uses only a small set of features and **one prediction snapshot per split**; it is for pipeline smoke testing, not final ranking evaluation.
- The popular-item negative pool introduces sampling bias; Notebook 03+ will construct proper candidate sets and assess retrieval Recall@K and ranking metrics on held-out positives.
- `known_property_count` measures metadata coverage, not semantic item content. Later notebooks can add property-value embeddings and category ancestry features.

**Next:** Upload `data/reports/notebook02_integration_temporal_report.json` to review actual coverage and splits, then proceed to Notebook 03.